# Simple LangChain Agent — walkthrough

This notebook is the **learning interface** of the project. The reusable code
lives in the `src/simple_agent` package; here we drive it step by step:

1. Load and validate the configuration
2. Test the Tavily search tool
3. Test the weather tool
4. Initialize the Groq LLM
5. Create the agent
6. Run the agent
7. Inspect the agent's message flow
8. Error handling

Run the cells from top to bottom with `uv run jupyter notebook`.

In [ ]:
from simple_agent import (
    create_agent,
    get_llm,
    get_search_tool,
    get_weather,
    load_config,
    missing_keys,
)

load_config()
print("Config loaded. Missing keys:", missing_keys() or "none — all set.")

## 2. Test Tavily on its own

Before involving the agent, check the tool directly. A *tool* is just a
function the LLM is allowed to call; `TavilySearch` hits the Tavily API and
returns clean, ranked snippets instead of raw HTML.

In [ ]:
search_tool = get_search_tool()
results = search_tool.invoke({"query": "What is the capital of France?"})
results

## 3. Test the weather tool on its own

`get_weather` calls the WeatherStack API and returns a one-line summary.
Note how failures (bad key, unknown city) come back as **plain strings** —
that is deliberate: returning a readable message lets the model recover
instead of crashing the agent loop.

In [ ]:
print(get_weather.invoke({"city": "Paris"}))
print(get_weather.invoke({"city": "Atlantis"}))  # graceful failure

## 4. Initialize the LLM

`get_llm()` returns a `ChatGroq` model (`openai/gpt-oss-120b`,
`temperature=0`, `max_retries=3`). Temperature 0 keeps tool selection
reproducible.

In [ ]:
llm = get_llm()
response = llm.invoke("What is the capital of Austria?")
print(response.content)

## 5–6. Create the agent and run it

`create_agent()` builds a LangGraph graph around the LLM, the two tools,
and a system prompt. The loop is: ask the model what to do → execute a tool
if requested → feed the result back → repeat until the model answers.

In [ ]:
agent = create_agent()

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Find the current weather in Paris and the capital of Austria",
            }
        ]
    }
)

print(result["messages"][-1].content)

## 7. Inspect the message flow

`result["messages"]` is the full trajectory: the user message, each
`AIMessage` with its tool calls, the `ToolMessage` results, and the final
answer. This is the first place to look when the agent does something odd.

In [ ]:
for message in result["messages"]:
    kind = message.type
    preview = message.content[:120].replace("\n", " ")
    print(f"[{kind}] {preview}")
    if getattr(message, "tool_calls", None):
        for call in message.tool_calls:
            print(f"    -> tool call: {call['name']}({call['args']})")

## 8. Error handling

Missing keys fail fast with a clear message instead of an opaque 401 from
the API. Temporarily unset a key to see it (then re-run the config cell):

In [ ]:
import os

from simple_agent import require_keys

saved = os.environ.pop("GROQ_API_KEY", None)
try:
    require_keys(["GROQ_API_KEY"])
except ValueError as exc:
    print(f"Caught expected error: {exc}")
finally:
    if saved is not None:
        os.environ["GROQ_API_KEY"] = saved